In [ ]:
!pip install transformers accelerate torch bitsandbytes pillow --quiet
!pip install rouge-score bert-score nltk openai-clip torch torchvision -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 34.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00


In [ ]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import torch.nn as nn
import math
import os
import threading
from functools import partial
from typing import Any, Dict, Optional, Tuple, List
from torch.optim import Optimizer
from PIL import Image
from io import BytesIO
import requests
from huggingface_hub import hf_hub_download

import re
import numpy as np
from transformers import EvalPrediction, BitsAndBytesConfig
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import f1_score
import nltk
nltk.download('punkt', quiet=True)


from datasets import load_dataset, Features, Value, Sequence
from transformers import (
    Qwen3VLForConditionalGeneration,
    MllamaForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model

In [ ]:
from huggingface_hub import whoami
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print(whoami(token=HF_TOKEN))

{'type': 'user', 'id': '65a19678974f8f811f578b05', 'name': 'mahdiislam79', 'fullname': 'Mahdi Islam', 'isPro': False, 'avatarUrl': '/avatars/ad6b84899444fa3d26fc26ab9906f843.svg', 'orgs': [{'type': 'org', 'id': '65b5db8428ec1c86aa27cd2f', 'name': 'BanglaLLM', 'fullname': 'Bangla Large Language Model', 'avatarUrl': 'https://www.gravatar.com/avatar/8f37fdd9943c5076695b8f656ba98417?d=retro&size=100', 'isEnterprise': False}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'my_token_1', 'role': 'fineGrained', 'createdAt': '2025-11-14T16:28:56.904Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '65a19678974f8f811f578b05', 'type': 'user', 'name': 'mahdiislam79'}, 'permissions': ['repo.content.read', 'repo.write', 'inference.endpoints.infer.write', 'inference.endpoints.write', 'inference.serverless.write']}]}}}}


In [ ]:
import os
os.chdir('/content/drive/MyDrive/GRIT/GRIT/Llama/')

print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/GRIT/GRIT/Llama


In [ ]:
import sys

# Add the current directory to the Python path
if './' not in sys.path:
    sys.path.insert(0, './')

from grit_llama import *
from data_utils_LLaVA_Llama import *
from metrics_llama import *

In [ ]:
def main():
    # === 1. Configuration Setup ===
    # Initialize the configuration object. This likely holds default values
    # for learning rate, model ID, batch size, etc.
    config = GRITConfig()

    model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
    config.model_id = model_id

    config.dataset_name = "liuhaotian/LLaVA-Instruct-150K"
    config.images_repo = getattr(config, "images_repo", "liuhaotian/LLaVA-Pretrain")
    config.image_cache_dir = getattr(config, "image_cache_dir", "./vqa_images_cache")
    config.image_search_folders = getattr(config, "image_search_folders", None)

    # === 2. GRIT-Specific Hyperparameters ===
    config.kfac_min_samples = 256        # Min samples before K-FAC calculates
    config.kfac_update_freq = 150        # How often (in steps) to update K-FAC
    config.kfac_damping = 0.005          # A stabilization term for K-FAC
    config.reprojection_warmup_steps = 500 # Don't start GRIT's rank magic immediately
    config.rank_adaptation_start_step = 500 # When to start adapting the LoRA ranks
    config.reprojection_freq = 300       # How often (in steps) to reassess adapter ranks
    config.use_two_sided_reprojection = True # A specific setting for the GRIT algorithm

    print(f"Loading processor and model for {model_id}...")

    # === 3. Load Model Processor ===
    # A "processor" for a VLM conveniently bundles:
    # 1. The Text Tokenizer: Turns text ("What color?") into numbers (input_ids).
    # 2. The Image Processor: Resizes, crops, and normalizes images for the model.
    processor = AutoProcessor.from_pretrained(
        model_id, # The ID of the model (e.g., "Qwen/Qwen2-7B-VL-Instruct")
        token=HF_TOKEN,
        trust_remote_code=True,     # Required for many new models on the Hub
    )

    # CRITICAL: Ensure the tokenizer has a pad token.
    # This is needed to make all sequences in a batch the same length
    # by adding a [PAD] token. If it doesn't have one, we set it to the
    # End-of-Sentence (eos) token.
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    # `partial` creates a new function `tok` that is just `tokenize_vqa`
    # but with the `config` and `processor` arguments already filled in.
    # This makes it easy to use with the dataset's `.map()` method.
    tok = partial(tokenize_vqa, config=config, processor=processor)

    model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

    bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
    )

    # === 4. Load the Base Model ===
    # This downloads and loads the massive, pre-trained Qwen3-VL model.
    model = MllamaForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN
    )

    # === 5. Configure LoRA (Low-Rank Adaptation) ===
    #
    #
    # Define which parts of the model we want to attach LoRA adapters to.
    # These are typically the 'linear' or 'attention' layers.
    target_modules = config.lora_target_modules.copy()
    # Optionally, also apply LoRA to the vision-encoder part of the model.
    if config.apply_lora_to_vision:
        target_modules.extend(config.vision_lora_modules)

    # Create the LoRA configuration object.
    lora_config = LoraConfig(
        r=config.lora_rank,             # 'r' is the "rank" or size of the adapter.
                                        # A key hyperparameter.
        lora_alpha=config.lora_alpha,   # A scaling factor for the adapter.
        target_modules=target_modules,  # The list of layers to adapt.
        lora_dropout=config.lora_dropout, # Dropout for regularization.
        bias="none",                    # A common LoRA setting.
        task_type="CAUSAL_LM",          # Specifies the model type for PEFT.
        inference_mode=False,           # We are in training mode.
        use_rslora=True,                # Uses Rank-Stabilized LoRA, an improved version.
    )

    # === 6. Apply LoRA to the Model ===
    # This is the magic step: it finds all layers in `target_modules`
    # and "injects" the trainable LoRA adapters into them.
    model = get_peft_model(model, lora_config)

    # Print a report on trainable vs. total parameters.
    # This proves the efficiency: trainable params will be a *tiny*
    # fraction (e.g., 0.1%) of the total params.
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    # This is required when using gradient checkpointing with LoRA.
    model.enable_input_require_grads()
    # Disable caching, which is useful for inference but problematic for training.
    model.config.use_cache = False

    # ============================================================
    # ADD THE DEVICE VALIDATION CHECK HERE
    # ============================================================
    print("\n" + "="*80)
    print("VALIDATING MODEL DEVICE PLACEMENT")
    print("="*80)

    meta_device_params = []
    for name, param in model.named_parameters():
        if param.device.type == 'meta':
            meta_device_params.append(name)
            print(f"⚠️  WARNING: Parameter '{name}' is on meta device!")

    if meta_device_params:
        print(f"\n❌ ERROR: Found {len(meta_device_params)} parameters still on meta device!")
        print("This will cause 'expected device meta but got cuda:0' errors during training.")
        print("\nParameters on meta device:")
        for i, param_name in enumerate(meta_device_params[:10], 1):  # Show first 10
            print(f"  {i}. {param_name}")
        if len(meta_device_params) > 10:
            print(f"  ... and {len(meta_device_params) - 10} more")
        raise RuntimeError(
            f"Model not fully materialized. {len(meta_device_params)} parameters "
            f"are still on meta device. Check your device_map configuration."
        )

    print("✅ All model parameters properly placed on device")
    print("="*80 + "\n")

    # ============================================================
    # CONTINUE WITH DATASET PROCESSING
    # ============================================================

    # === 7. Load and Prepare the Dataset ===

    # Load the LLaVA instruction dataset from its JSON file on the Hub.
    llava = load_dataset(
        "json",
        data_files="https://huggingface.co/datasets/liuhaotian/LLaVA-Instruct-150K/resolve/main/llava_instruct_150k.json",
        split="train", # We are loading the 'train' split
    )

    # NOTE: Selecting a very small subset for a quick demonstration/debug run.
    # For a real training run, you would use much larger numbers.
    train_dataset = llava.shuffle(seed=42).select(range(200))
    val_dataset   = llava.shuffle(seed=42).select(range(50))
    test_dataset  = llava.shuffle(seed=42).select(range(50))

    print(f"Final splits → Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    print("Tokenizing datasets...")
    print("Processing training data...")

    # === 8. Tokenize the Datasets ===
    # Use the `.map()` function to apply our `tok` (tokenize_vqa) function
    # to every single example in the dataset.
    tokenized_train_dataset = train_dataset.map(
        tok, # The function we defined earlier with `partial`
        remove_columns=train_dataset.column_names, # Delete old columns
        num_proc=2, # Number of parallel processes to use
        desc="Processing training data",
        load_from_cache_file=False # Force re-processing
    )

    print("Filtering valid training samples...")
    # The `tokenize_vqa` function might return `None` if an image fails to
    # load or if data is corrupt. This filter removes those bad samples.
    tokenized_train_dataset = tokenized_train_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering training data"
    )

    # Repeat the same tokenization and filtering process for validation...
    print("Processing validation data...")
    tokenized_val_dataset = val_dataset.map(
        tok,
        remove_columns=val_dataset.column_names,
        num_proc=1,
        desc="Processing validation data",
        load_from_cache_file=False
    )
    print("Filtering valid validation samples...")
    tokenized_val_dataset = tokenized_val_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering validation data"
    )

    # ...and test datasets.
    print("Processing test data...")
    tokenized_test_dataset = test_dataset.map(
        tok,
        remove_columns=test_dataset.column_names,
        num_proc=1,
        desc="Tokenizing test",
        load_from_cache_file=False,
    )
    print("Filtering valid test samples...")
    tokenized_test_dataset = tokenized_test_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering test data"
    )

    # Get the final counts after filtering.
    train_len = len(tokenized_train_dataset)
    val_len = len(tokenized_val_dataset)
    test_len = len(tokenized_test_dataset)

    # A crucial sanity check. If no data survived, stop the script.
    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    print(f"\nSuccessfully processed datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    # === 9. Calculate Training Steps ===
    # "Effective Batch Size" is the batch size per device * gradient accumulation steps.
    eff_batch = config.batch_size * config.gradient_accumulation_steps
    # Total steps = (samples / batch_size) * epochs
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    # === 10. Configure the Trainer (TrainingArguments) ===
    # This object holds all the training settings (learning rate, epochs, etc.)
    training_args = Seq2SeqTrainingArguments(
        output_dir="./results_vqa_grit_llama3_2_vision", # Where to save checkpoints
        per_device_train_batch_size=config.batch_size, # Batch size per GPU
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps, # Accumulate gradients to simulate a larger batch size
        num_train_epochs=config.num_epochs, # Total number of times to go over the data
        learning_rate=config.learning_rate, # The optimizer's learning rate
        bf16=(config.precision == "bf16"),  # Use bf16 mixed precision (if supported)
        fp16=(config.precision == "fp16"),  # Use fp16 mixed precision
        max_grad_norm=1.5,                  # Gradient clipping to prevent explosions
        gradient_checkpointing=True,        # A key memory-saving technique
        gradient_checkpointing_kwargs={"use_reentrant": False}, # Modern setting
        dataloader_num_workers=config.num_workers, # Parallel data loading
        dataloader_pin_memory=config.pin_memory,
        dataloader_drop_last=config.drop_last,
        remove_unused_columns=False,    # We need to keep 'image' column for collate_fn
        report_to="none",               # Disable reporting to WandB/etc.
        save_strategy="epoch",          # Save a checkpoint every epoch
        eval_strategy="epoch",          # Evaluate on the validation set every epoch
        save_total_limit=2,             # Only keep the last 2 checkpoints
        logging_steps=50,               # Log metrics every 50 steps
        warmup_steps=100,               # Slowly ramp up the learning rate
        optim="adamw_torch",            # Use the AdamW optimizer
        load_best_model_at_end=True,    # Reload the best model (by val loss) at the end
        metric_for_best_model="loss",
    )



    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nInitializing GRIT on device: {device}")

    # === 11. Initialize GRIT Components ===
    # Create the GRITManager, the "brain" that will handle rank adaptation.
    grit_manager = GRITManager(model, config, device)

    # Create the data collator function. This function takes a list of
    # individual samples (like `[sample1, sample2, sample3]`) and
    # intelligently batches them into a single tensor, handling padding.
    collate = partial(collate_fn, config=config, processor=processor)

    # === 12. Initialize the Custom GritTrainer ===
    # We use `GritTrainer` (not the standard `Trainer`) because it's
    # built to work with the `GRITManager`.
    trainer = GritTrainer(
        processor = processor,
        grit_manager=grit_manager, # Pass in the GRIT manager
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate, # Use our custom collate function
        compute_metrics=make_compute_metrics(processor), # Function to calculate ROUGE, etc.
        callbacks=[GritCallback(grit_manager)], # The "link" between the trainer and manager
        # Add test for final evaluation
        # test_dataset=tokenized_test_dataset,
    )

    # Print a final summary of the training run.
    print("\n" + "="*80)
    print("STARTING GRIT TRAINING FOR LLAMA 3.2 VISION MODEL")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"LoRA rank: {config.lora_rank}")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"K-FAC update freq: {config.kfac_update_freq} steps")
    print(f"Reprojection freq: {config.reprojection_freq} steps")
    print(f"Vision LoRA: {'Enabled' if config.apply_lora_to_vision else 'Disabled'}")
    print(f"Rank adaptation: {'Enabled' if config.enable_rank_adaptation else 'Disabled'}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print("="*80 + "\n")

    # === 13. Run Training and Evaluation ===
    try:
        print("Beginning training...")
        # This is the main command that starts the entire training process.
        trainer.train()

        print("\n" + "="*80)
        print("FINAL TEST EVALUATION")
        print("="*80)

        # After training, run a final evaluation on the held-out test set.
        test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset)

        # Print the final scores.
        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR        : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1  : {test_results['eval_bertscore_f1']:.4f}")

        print("\n" + "="*80)
        print("Saving final model...")
        output_dir = "./final_vqa_grit_llama3_2_vision_model"
        # This saves only the trained LoRA adapter weights, not the full model.
        # This is great! The adapters are tiny (megabytes) vs. the base model (gigabytes).
        model.save_pretrained(output_dir)
        # Also save the processor so you can easily reload everything later.
        processor.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    # === 14. Error and Interruption Handling ===
    except KeyboardInterrupt:
        # This 'except' block runs if you manually stop the script (e.g., Ctrl+C).
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            # Try to save the current state so you can resume later.
            output_dir = "./interrupted_checkpoint_llama3_2_vision"
            model.save_pretrained(output_dir)
            processor.save_pretrained(output_dir)
            print(f"Checkpoint saved to {output_dir}")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        # This 'except' block runs if any other error occurs (e.g., out-of-memory).
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc() # Print the full error details
        raise

    finally:
        # This 'finally' block *always* runs, whether the training
        # succeeded, failed, or was interrupted.
        print("\nCleaning up...")
        # Clear the custom image cache
        _image_cache = LazyImageCache()
        _image_cache.clear()
        # Run Python's garbage collector to free up memory
        gc.collect()
        # If using a GPU, explicitly tell PyTorch to empty its VRAM cache.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


# This is the standard Python entry point.
# If the script is run directly (e.g., `python train.py`),
# it will call the `main()` function.
if __name__ == "__main__":
    main()

Loading processor and model for meta-llama/Llama-3.2-11B-Vision-Instruct...


preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...


Successfully processed datasets:
Training samples: 200
Validation samples: 50
Test samples: 50

Training Configuration:
Total training steps: 75
Effective batch size: 8
Steps per epoch: 25

Initializing GRIT on device: cuda
Instrumenting model for GRIT...
Instrumented 0 vision modules + 520 language modules
Total LoRA modules with K-FAC: 520
GRITManager: Initialization complete for Llama-3.2-11B model.
Optimizing 520 LoRA modules across vision + language towers.
GritTrainer: Initialized with GRIT for Llama 3.2 model.

STARTING GRIT TRAINING FOR LLAMA 3.2 VISION MODEL
Model: meta-llama/Llama-3.2-11B-Vision-Instruct
Dataset: liuhaotian/LLaVA-Instruct-150K
LoRA rank: 8
Batch size: 1 x 8 = 8
Learning rate: 1e-05
Precision: bf16
K-FAC update freq: 150 steps
Reprojection freq: 300 steps
Vision LoRA: Disabled
Rank adaptation: Enabled
Training samples: 200
Validation samples: 50
Test samples: 50

Beginning training...
Wrapping the optimizer with GRIT preconditioning logic.


Epoch,Training Loss,Validation Loss,Rouge L,Meteor,Bertscore F1
1,No log,1.445648,0.000000,0.000000,0.000000



Clearing VRAM before evaluation...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



EVALUATION SAMPLES

[Sample 1]
Generated: 
Reference: In this bustling city scene, people are walking along the sidewalk near a block of store shops. Amon

[Sample 2]
Generated: 
Reference: This meal appears to be well balanced and nutritious, as it includes a variety of food items such as

[Sample 3]
Generated: 
Reference: The man's work setup is different from a typical office environment as he is using his laptop while 


Metrics Statistics:
  Total samples: 50
  Empty predictions: 50
  Empty references: 0
  Valid pairs for scoring: 0
